# Calibration Exercise — Gwet's AC1 & Confusion Index

**Purpose:** Before reference data collection begins, this notebook checks whether interpreters
are applying the response design consistently. It combines each interpreter's individual CSV
into one dataset, then runs two complementary calibration diagnostics for three variables
(**LULC class**, **LULC subclass**, and **Change/No-Change**):

1. **Inter-Rater Agreement (Gwet's AC1)** — pairwise agreement between every pair of interpreters,
   corrected for chance agreement using overall class prevalence, with bootstrap confidence
   intervals and a heatmap per variable.
2. **Confusion Index** — which classes interpreters most often disagree on, as a bar chart per
   variable, to target follow-up training.

**Why Gwet's AC1 specifically:** Unlike Cohen's Kappa, Gwet's AC1 estimates chance agreement from
the average prevalence of each class across both raters, rather than from each rater's individual
labeling habits. This makes it stable even when one class strongly dominates a variable (e.g.
Change/No-Change, where "No Change" usually dominates) -- Kappa can understate agreement in that
situation even for accurate interpreters, a known limitation called the "Kappa paradox". For
reasonably balanced variables (e.g. LULC, subclass here), the companion `Calibration_Kappa.ipynb`
notebook is the more standard, literature-comparable choice. A third companion,
`Calibration_Jaccard.ipynb`, is the most sensitive of the three to a single rare class's problems
specifically, and breaks down per class rather than as one overall number. All three share the
same data-prep, Confusion Index, and export structure, so results are directly comparable across
notebooks.

**How to use this notebook:** Only edit **Section 1 — Configuration** below. Everything after that
runs automatically off those settings. Run cells top to bottom.

**Outputs produced:**
- `operator_identity_mapping.csv` — background file linking anonymized `operator_XX` IDs to real
  interpreter names. Kept separate from all results so results can be shared without revealing
  who interpreted what.
- `calibration_results_gwet.xlsx` — one workbook, one sheet per result table (Gwet's AC1 and
  Confusion Index, for each of the three variables), plus a summary sheet.


## 1. Configuration

**This is the only section you should need to edit.** Update the paths and column names below to
match your project, then run the whole notebook.


In [ ]:
# ---- Folders ----
INPUT_FOLDER = "/home/sepal-user/Country_support/Benin/MIA/CSV/"   # folder containing one CSV per interpreter
OUTPUT_FOLDER = "/home/sepal-user/Country_support/Benin/MIA/Calibration_Outputs/"  # all outputs are written here

# ---- Identity columns (as they appear in each raw interpreter CSV) ----
RAW_ID_COLUMN = "id"                # unique sample/plot identifier
RAW_OPERATOR_NAME_COLUMN = None     # set to the column name if the raw CSV already has a real operator
                                     # name column (e.g. "operator"); leave as None if identity should
                                     # only come from the filename

# ---- Location columns ----
# Not used in calibration math, but carried through so this output plugs directly into the
# quality-control stage later. Different collection tools name these differently -- just retype
# the three names below to match whichever tool produced this data. Nothing else needs to change.
ID_COLUMN = "id"            # e.g. "id" or "plotid"
X_COLUMN = "location_x"     # e.g. "location_x" or "XCoordinate" or "Central_long"
Y_COLUMN = "location_y"     # e.g. "location_y" or "YCoordinate" or "Central_lat"

# ---- Class columns to analyse ----
LULC_COLUMN = "lu_t2"           # land-use/land-cover class column
SUBCLASS_COLUMN = "sub_t2"      # LULC subclass column
CHNCH_COLUMN = "CHnCH"          # change / no-change column

# ---- Change/No-Change derivation (used ONLY if CHNCH_COLUMN is not already present in the data) ----
# Calibration only cares about one direction of change (e.g. Forest -> non-Forest); QC (a later,
# separate stage) is where broader change detection happens.
CHNCH_FROM_LU_COLUMN = "event[1]_from_lu"     # class the sample was originally labelled
CHNCH_TO_LU_COLUMN = "lu_t2_label"            # class the sample is labelled now
CHNCH_FROM_CLASS = "F"                        # the "from" class that counts as a change (e.g. Forest)
CHNCH_YEAR_COLUMN = "event[1]_change_yr"      # year the change event was recorded
CHNCH_YEAR_THRESHOLD = 2014                   # only changes on/after this year count

# ---- Variables to run through this notebook's metric AND Confusion Index ----
# (label, column name) -- add/remove/reorder freely; everything below adapts automatically
VARIABLES = [
    ("LULC", LULC_COLUMN),
    ("LULC_subclass", SUBCLASS_COLUMN),
    ("CHnCH", CHNCH_COLUMN),
]

# ---- Statistics ----
N_BOOTSTRAP = 1000     # bootstrap resamples for confidence intervals
ALPHA = 0.05           # 95% CI
RANDOM_SEED = 42       # for reproducibility

# ---- Anonymized operator ID format ----
# Interpreter identity is deliberately hidden from all analysis outputs and only recoverable via
# the separate identity mapping file. Operators are labelled "operator 01", "operator 02", ... in
# the order their files are read (sorted by filename for reproducibility).
OPERATOR_ID_PREFIX = "operator "

# ---- Optional: timestamp column (for vigilance/fatigue check in Section 4) ----
# Only one of your two source tools records this. Leave as None if not available for this
# dataset -- Section 4's vigilance check will simply be skipped, no error.
TIMESTAMP_COLUMN = None       # e.g. "interpretation_time" -- set to the real column name if present

# ---- Optional: confidence/difficulty column (for confidence cross-tab in Section 4) ----
# Leave as None if interpreters don't flag "hard" / "low confidence" samples in this dataset.
CONFIDENCE_COLUMN = None      # e.g. "confidence_level"

# ---- Class colors for confusion-index bar charts ----
# Keyed by class code AND common full names (case-insensitive lookup), so this works whether your
# data uses "F" or "Forest", etc. Add more aliases here if your classification system differs.
CLASS_COLOR_MAP = {
    "f": "#228B22", "forest": "#228B22",              # green
    "g": "#FF8C00", "grassland": "#FF8C00",            # orange
    "c": "#FFD54F", "cropland": "#FFD54F",              # yellow
    "s": "#FF0000", "settlement": "#FF0000", "settlements": "#FF0000",  # red
    "w": "#1E90FF", "wetland": "#1E90FF",               # blue
    "o": "#A9A9A9", "otherland": "#A9A9A9", "other": "#A9A9A9",         # grey
}


### 1.1 Setup

Imports and output folder creation. No need to edit this.


In [ ]:
import os
import warnings
from glob import glob
import itertools
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
sns.set(style="whitegrid")


def operator_sort_key(op):
    """Sorts operator labels in natural numeric order (operator 1, 2, ... 10, 11 --
    not the alphabetical 1, 10, 11, ... 2 that plain string sorting would give)."""
    import re
    m = re.search(r"(\d+)\s*$", str(op))
    return int(m.group(1)) if m else str(op)


def add_rank_column(df, value_col, worst_is_high, rank_name="Rank (most concern first)"):
    """Adds a rank column (1 = needs attention first) without reordering the table --
    the table itself stays in operator ID order for easy lookup."""
    if df.index.duplicated().any():
        dupes = sorted(set(df.index[df.index.duplicated()]))
        print(f"  WARNING: operator ID(s) {dupes} appeared more than once -- likely a duplicate "
              f"or backup CSV file for that interpreter in the input folder. Averaging the "
              f"duplicate rows so the notebook can continue; check Section 1.2's file list to "
              f"find and remove the extra file, then rerun.")
        df = df.groupby(level=0).mean(numeric_only=True)
    df = df.reindex(sorted(df.index, key=operator_sort_key))
    ranked = df[value_col].rank(ascending=worst_is_high, method="min").astype(int)
    df.insert(0, rank_name, ranked)
    return df


def plot_operator_bar(df, value_col, title, save_path, worst_is_high, flag_fraction=0.25):
    """Horizontal bar chart, ranked worst-first, for quick visual scanning -- the printed table
    stays in operator-ID order, this chart is the fast-read companion to it. The worst
    `flag_fraction` of operators are highlighted in red."""
    plot_df = df.sort_values(value_col, ascending=not worst_is_high)
    n_flag = max(1, int(round(len(plot_df) * flag_fraction)))
    colors = ["#D62728" if i < n_flag else "#4C72B0" for i in range(len(plot_df))]

    plt.figure(figsize=(9, max(3, 0.45 * len(plot_df))))
    plt.barh(plot_df.index.astype(str), plot_df[value_col], color=colors)
    plt.gca().invert_yaxis()  # highest-concern operator at the top
    plt.xlabel(value_col, fontsize=12, fontweight="bold")
    plt.title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def class_color(label, fallback_palette=itertools.cycle(sns.color_palette("tab10").as_hex())):
    """Look up a fixed color for a class label (case-insensitive; matches code or full name).
    Falls back to a cycling default palette for classes not in CLASS_COLOR_MAP, with a one-time note."""
    key = str(label).strip().lower()
    if key in CLASS_COLOR_MAP:
        return CLASS_COLOR_MAP[key]
    color = next(fallback_palette)
    print(f"  Note: no configured color for class '{label}' -- using a default color. "
          f"Add it to CLASS_COLOR_MAP in Section 1 to fix its color permanently.")
    return color


def sanitize_filename(text):
    return "".join(c if c.isalnum() else "_" for c in str(text))


def plot_agreement_heatmap(results_df, value_col, title, save_path, negative_color="#C0392B",
                            annot_color="black"):
    raters = sorted(set(results_df["Operator 1"]) | set(results_df["Operator 2"]), key=operator_sort_key)
    mat = pd.DataFrame(np.nan, index=raters, columns=raters)
    labels = pd.DataFrame("", index=raters, columns=raters)

    for _, row in results_df.iterrows():
        r1, r2 = row["Operator 1"], row["Operator 2"]
        if raters.index(r1) > raters.index(r2):
            mat.loc[r1, r2] = row[value_col]
            labels.loc[r1, r2] = f"{row[value_col]:.2f}"
        else:
            mat.loc[r2, r1] = row[value_col]
            labels.loc[r2, r1] = f"{row[value_col]:.2f}"

    # Fixed 0-1 color scale on every heatmap, regardless of this dataset's actual range, so
    # heatmaps from different variables/projects are visually comparable at a glance. Values
    # below 0 (worse than chance agreement) are drawn in a distinct solid color rather than being
    # folded into the 0-1 scale, so they're visually obvious rather than looking identical to 0.
    upper_mask = np.triu(np.ones_like(mat, dtype=bool), k=0)
    negative_mask = upper_mask | mat.isna() | (mat >= 0)
    nonneg_mask = upper_mask | mat.isna() | (mat < 0)

    plt.figure(figsize=(max(9, len(raters) + 3.5), max(7, len(raters))))
    ax = plt.gca()
    # Fixed single annotation color (standard protocol) rather than automatic black/white contrast
    # per cell. Trade-off: legibility can dip on the very darkest cells (values approaching 1.0).
    sns.heatmap(mat, annot=labels, fmt="", cmap="YlGnBu", mask=nonneg_mask, vmin=0, vmax=1,
                square=True, linewidths=0.5, linecolor="white", cbar_kws={"label": value_col},
                annot_kws={"color": annot_color}, ax=ax)
    if (~negative_mask).any().any():
        from matplotlib.colors import ListedColormap
        sns.heatmap(mat, annot=labels, fmt="", cmap=ListedColormap([negative_color]), mask=negative_mask,
                    square=True, linewidths=0.5, linecolor="white", cbar=False,
                    annot_kws={"color": annot_color}, ax=ax)
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(facecolor=negative_color, label=f"{value_col} < 0 (worse than chance)")],
                  loc="lower left", bbox_to_anchor=(0, -0.32), frameon=False, fontsize=11)

    plt.title(title, fontsize=15, pad=14)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


print(f"Outputs will be written to: {OUTPUT_FOLDER}")


### 1.2 Combine interpreter files & assign anonymized IDs

Read every CSV in the input folder, assign each interpreter an anonymized ID
(`operator_01`, `operator_02`, ...), and build the identity mapping (kept separate from results).


In [ ]:
csv_files = sorted(glob(os.path.join(INPUT_FOLDER, "*.csv")))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {INPUT_FOLDER}. Check INPUT_FOLDER in Step 1."
    )

print(f"Found {len(csv_files)} interpreter file(s):")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")

raw_frames = {}          # anonymized_id -> dataframe
identity_records = []    # for the mapping file

for i, file in enumerate(csv_files, start=1):
    anon_id = f"{OPERATOR_ID_PREFIX}{i:02d}"
    real_name = os.path.splitext(os.path.basename(file))[0]
    if RAW_OPERATOR_NAME_COLUMN:
        df = pd.read_csv(file)
        if RAW_OPERATOR_NAME_COLUMN in df.columns and df[RAW_OPERATOR_NAME_COLUMN].nunique() == 1:
            real_name = str(df[RAW_OPERATOR_NAME_COLUMN].iloc[0])
    else:
        df = pd.read_csv(file)

    df["operator"] = anon_id
    raw_frames[anon_id] = df
    identity_records.append({"operator": anon_id, "real_name": real_name, "source_file": os.path.basename(file)})

identity_df = pd.DataFrame(identity_records)
identity_path = os.path.join(OUTPUT_FOLDER, "operator_identity_mapping.csv")
identity_df.to_csv(identity_path, index=False)

print(f"\nIdentity mapping saved separately (background file, not shared with results):\n  {identity_path}")
identity_df


### 1.3 Check sample completeness across operators

Check that every operator's file covers the same set of samples. If a sample ID is
missing from any operator's file, it is dropped from **all** operators before analysis, so every
pairwise comparison is on a common, complete sample set. This is a completeness check, not a
duplicate check — in calibration data, each sample ID is *expected* to repeat once per operator
(e.g. 100 samples x 20 operators = 2,000 rows).


In [ ]:
row_counts = {op: len(df) for op, df in raw_frames.items()}
print("Rows per operator file:")
for op, n in row_counts.items():
    print(f"  {op}: {n}")

id_sets = {op: set(df[RAW_ID_COLUMN]) for op, df in raw_frames.items()}
common_ids = set.intersection(*id_sets.values())

print(f"\nSamples present in ALL operator files: {len(common_ids)}")

for op, ids in id_sets.items():
    missing = ids - common_ids
    if missing:
        print(f"  WARNING: {op} has {len(missing)} sample(s) not shared by all operators -- these will be excluded: {sorted(missing)[:10]}{'...' if len(missing) > 10 else ''}")

if len(common_ids) == 0:
    raise ValueError("No sample IDs are common to all operator files -- check RAW_ID_COLUMN and the input data.")

frames_aligned = []
for op, df in raw_frames.items():
    df_aligned = df[df[RAW_ID_COLUMN].isin(common_ids)].copy()
    frames_aligned.append(df_aligned)

combined_df = pd.concat(frames_aligned, ignore_index=True, join="outer")
print(f"\nCombined dataset: {len(combined_df)} rows across {len(raw_frames)} operators and {len(common_ids)} common samples.")


### 1.4 Verify or derive the Change/No-Change column

If `CHNCH_COLUMN` already exists in the data, it's used as-is. If not, it's derived here (once,
in the data-prep stage) using the rule set in Section 1 — never mid-analysis.


In [ ]:
if CHNCH_COLUMN in combined_df.columns:
    print(f"'{CHNCH_COLUMN}' already present in the data -- using as-is.")
else:
    print(f"'{CHNCH_COLUMN}' not found -- deriving it from Step 1 rules "
          f"({CHNCH_FROM_LU_COLUMN} == '{CHNCH_FROM_CLASS}' and {CHNCH_TO_LU_COLUMN} != '{CHNCH_FROM_CLASS}' "
          f"and {CHNCH_YEAR_COLUMN} >= {CHNCH_YEAR_THRESHOLD}).")
    combined_df[CHNCH_COLUMN] = np.where(
        (combined_df[CHNCH_FROM_LU_COLUMN] == CHNCH_FROM_CLASS) &
        (combined_df[CHNCH_TO_LU_COLUMN] != CHNCH_FROM_CLASS) &
        (combined_df[CHNCH_YEAR_COLUMN] >= CHNCH_YEAR_THRESHOLD),
        "Change", "No Change",
    )

print(combined_df[CHNCH_COLUMN].value_counts())


### 1.5 Reshape to wide format

Each variable (LULC, LULC subclass, CHnCH) is pivoted so rows = samples and columns = operators.
This is the shape this notebook's metric and Confusion Index analysis need. Operator columns are
detected automatically -- no need to know or hardcode how many interpreters there are.


In [ ]:
def reshape_wide(df, value_col, id_col=RAW_ID_COLUMN, operator_col="operator"):
    wide = df.pivot_table(
        index=id_col,
        columns=operator_col,
        values=value_col,
        aggfunc=lambda x: x.unique()[0] if len(x.unique()) > 0 else None,
    ).reset_index()
    wide.columns.name = None
    return wide

wide_data = {}
for label, col in VARIABLES:
    wide_data[label] = reshape_wide(combined_df, col)
    print(f"{label}: {wide_data[label].shape[0]} samples x "
          f"{wide_data[label].shape[1] - 1} operators")


### 1.6 Sanity checks

A quick summary to confirm the data looks right before running any statistics -- including a
per-operator class-count table, so you can spot rare-class sparsity (e.g. very few "Change"
samples for a given operator) before it shows up as an undefined statistic later.


In [ ]:
print("=== Sanity check summary ===\n")
print(f"Operators: {len(raw_frames)}")
print(f"Common samples: {len(common_ids)}\n")

class_count_tables = {}   # label -> operator x class count table

for label, col in VARIABLES:
    wide = wide_data[label]
    operator_cols = [c for c in wide.columns if c != RAW_ID_COLUMN]
    classes_seen = sorted(pd.unique(wide[operator_cols].values.ravel()))
    classes_seen = [c for c in classes_seen if pd.notna(c)]
    print(f"[{label}] classes observed: {classes_seen}")

    counts = pd.DataFrame({op: wide[op].value_counts() for op in operator_cols}).fillna(0).astype(int).T
    counts.index.name = "operator"
    class_count_tables[label] = counts
    print(f"\nPer-operator class counts [{label}]:")
    print(counts)
    print()


## 2. Calibration: Inter-Rater Agreement (Gwet's AC1)

Chance-corrected agreement using overall class prevalence across both raters, rather than each
rater's own habits -- this avoids the "Kappa paradox" (Kappa understating agreement when one class
dominates). Shared functions used for all three variables, so the logic exists once.


In [ ]:
def gwet_ac1_score(l1, l2, categories):
    """Gwet's AC1: like Kappa, corrects observed agreement for chance -- but estimates chance
    agreement from the average category prevalence across both raters combined, rather than from
    each rater's individual marginals. This avoids the 'Kappa paradox', where Kappa can report low
    or even negative agreement for raters who are mostly correct, purely because one class (e.g.
    'No Change') dominates the data."""
    q = len(categories)
    if q < 2 or len(l1) == 0:
        return np.nan
    po = (l1 == l2).mean()
    pi_k = [((l1 == k).mean() + (l2 == k).mean()) / 2 for k in categories]
    pe = sum(p * (1 - p) for p in pi_k) / (q - 1)
    if pe >= 1:
        return np.nan
    return (po - pe) / (1 - pe)


def bootstrap_gwet(labels1, labels2, categories, n_bootstrap, rng):
    n = len(labels1)
    boot = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        boot[b] = gwet_ac1_score(labels1[idx], labels2[idx], categories)
    return boot


def pairwise_gwet(wide_df, categories, id_col=RAW_ID_COLUMN, n_bootstrap=N_BOOTSTRAP, alpha=ALPHA, rng=rng):
    df = wide_df.dropna().copy()
    operator_cols = [c for c in df.columns if c != id_col]

    results, all_boot = [], []
    for op1, op2 in itertools.combinations(operator_cols, 2):
        l1, l2 = df[op1].values, df[op2].values
        valid = ~(pd.isna(l1) | pd.isna(l2))
        l1, l2 = l1[valid], l2[valid]
        if len(l1) == 0:
            continue

        ac1_obs = gwet_ac1_score(l1, l2, categories)
        boot = bootstrap_gwet(l1, l2, categories, n_bootstrap, rng)
        ci_lo, ci_hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        po = (l1 == l2).mean()

        results.append({
            "Operator 1": op1, "Operator 2": op2,
            "Observed Agreement": round(po, 3),
            "Gwet's AC1": round(ac1_obs, 3),
            "CI Lower": round(ci_lo, 3), "CI Upper": round(ci_hi, 3),
        })
        all_boot.extend(boot)

    return pd.DataFrame(results), np.array(all_boot)


### 2.1 Pairwise Gwet's AC1 & heatmap


In [ ]:
gwet_results = {}   # label -> results dataframe

for label, col in VARIABLES:
    print(f"\n=== Gwet's AC1: {label} ===")
    wide = wide_data[label]
    operator_cols = [c for c in wide.columns if c != RAW_ID_COLUMN]
    categories = sorted(pd.unique(wide[operator_cols].values.ravel()))
    categories = [c for c in categories if pd.notna(c)]

    gdf, all_boot = pairwise_gwet(wide, categories)
    gwet_results[label] = gdf

    if gdf.empty:
        print(f"  No valid pairwise comparisons for {label} -- skipping.")
        continue

    mean_ac1 = gdf["Gwet's AC1"].mean()
    print(f"  Mean Gwet's AC1: {mean_ac1:.3f}")
    ci_lo, ci_hi = np.percentile(all_boot, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
    print(f"  95% CI: ({ci_lo:.3f}, {ci_hi:.3f})")

    plot_path = os.path.join(OUTPUT_FOLDER, f"gwet_heatmap_{label}.png")
    plot_agreement_heatmap(gdf, "Gwet's AC1", f"Inter-Rater Agreement (Gwet's AC1) — {label}", plot_path)


### 2.2 Per-operator agreement summary

The heatmap above shows agreement *between pairs*. This collapses it to one score *per operator*
-- their average Gwet's AC1 across every pairing they're part of -- so you can see which
interpreters are least aligned with the group. The table stays in operator-ID order for easy
lookup; the chart next to it is sorted worst-first for quick visual scanning, with the
lowest-agreement quarter of operators highlighted in red.


In [ ]:
operator_agreement_gwet = {}   # label -> per-operator mean AC1 dataframe

for label, _ in VARIABLES:
    gdf = gwet_results.get(label)
    if gdf is None or gdf.empty:
        continue

    long = pd.concat([
        gdf[["Operator 1", "Gwet's AC1"]].rename(columns={"Operator 1": "operator"}),
        gdf[["Operator 2", "Gwet's AC1"]].rename(columns={"Operator 2": "operator"}),
    ])
    per_op = long.groupby("operator")["Gwet's AC1"].agg(["mean", "count"]).round(3)
    per_op.columns = ["Mean Gwet's AC1", "Pairings"]
    per_op = add_rank_column(per_op, "Mean Gwet's AC1", worst_is_high=False)
    operator_agreement_gwet[label] = per_op

    print(f"\n[{label}] Per-operator mean Gwet's AC1:")
    print(per_op)

    bar_path = os.path.join(OUTPUT_FOLDER, f"gwet_by_operator_{label}.png")
    plot_operator_bar(per_op, "Mean Gwet's AC1", f"Mean Gwet's AC1 by Operator — {label}", bar_path, worst_is_high=False)


## 3. Confusion Index

Identifies which classes interpreters most often disagree on (pairwise, no ground truth needed —
appropriate here since calibration has no reference/"truth" labels). Shown as a bar chart per
variable rather than a heatmap, and includes a per-sample "how many unique labels did operators
give this sample" diagnostic — the same kind of check used to flag samples for group discussion.


In [ ]:
def pairwise_confusion(wide_df, id_col=RAW_ID_COLUMN):
    df = wide_df.copy()
    operator_cols = [c for c in df.columns if c != id_col]
    for c in operator_cols:
        df[c] = df[c].astype(str)

    all_labels = sorted(pd.unique(df[operator_cols].values.ravel()))
    all_labels = [l for l in all_labels if l not in ("nan", "None")]

    confusion_mats = {}
    for c1, c2 in itertools.combinations(operator_cols, 2):
        valid = df[c1].notna() & df[c2].notna() & (df[c1] != "nan") & (df[c2] != "nan")
        y_true, y_pred = df.loc[valid, c1], df.loc[valid, c2]
        cm = confusion_matrix(y_true, y_pred, labels=all_labels).astype(float)
        total = cm.sum()
        if total == 0:
            continue
        conf_idx = cm.copy()
        np.fill_diagonal(conf_idx, 0)
        conf_idx /= total
        confusion_mats[(c1, c2)] = pd.DataFrame(conf_idx, index=all_labels, columns=all_labels)

    if not confusion_mats:
        return None, all_labels, None

    avg_conf = sum(confusion_mats.values()) / len(confusion_mats)
    return avg_conf, all_labels, confusion_mats


def top_confused_classes(avg_conf, all_labels, top_n=3):
    records = []
    for cls in all_labels:
        row = avg_conf.loc[cls].drop(cls).sort_values(ascending=False).head(top_n)
        for confused_class, value in row.items():
            records.append({"Base Class": cls, "Confused With": confused_class, "Confusion Index (%)": round(value * 100, 2)})
    return pd.DataFrame(records)


def plot_confusion_bars(grouped_df, title, save_path, base_class_order):
    grouped_df = grouped_df.copy()
    grouped_df["Base Class"] = pd.Categorical(grouped_df["Base Class"], categories=base_class_order, ordered=True)
    grouped_df = grouped_df.sort_values("Base Class")

    confused_categories = list(dict.fromkeys(grouped_df["Confused With"]))  # first-seen order
    palette = {cat: class_color(cat) for cat in confused_categories}

    plt.figure(figsize=(max(10, len(base_class_order) * 2.5), 7))
    ax = sns.barplot(data=grouped_df, x="Base Class", y="Confusion Index (%)",
                      hue="Confused With", hue_order=confused_categories, palette=palette)

    for i in range(1, len(base_class_order)):
        plt.axvline(x=i - 0.5, color="gray", linestyle="--", linewidth=0.7)

    ax.set_title(title, fontsize=18, fontweight="bold")
    ax.set_xlabel("Base Class", fontsize=14, fontweight="bold")
    ax.set_ylabel("Confusion Index (%)", fontsize=14, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=12)
    plt.yticks(fontsize=12)
    ax.legend(title="Confused With", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
confusion_results = {}       # label -> top-confused-classes dataframe
confusion_matrices = {}      # label -> averaged pairwise confusion matrices (for 3.2)
sample_agreement_results = {}  # label -> per-sample unique-label-count dataframe

for label, col in VARIABLES:
    print(f"\n=== Confusion Index: {label} ===")
    avg_conf, all_labels, pairwise_mats = pairwise_confusion(wide_data[label])

    if avg_conf is None:
        print(f"  No valid pairwise comparisons for {label} -- skipping.")
        continue

    if len(all_labels) <= 2:
        print(f"  Only {len(all_labels)} class(es) present ({all_labels}) -- a confusion-index "
              f"chart isn't informative for a binary variable (it's just the mirror of itself). "
              f"Skipping the chart.")
    else:
        top_df = top_confused_classes(avg_conf, all_labels, top_n=3)
        confusion_results[label] = top_df
        confusion_matrices[label] = pairwise_mats
        print(top_df.head(10))

        bar_path = os.path.join(OUTPUT_FOLDER, f"confusion_index_{label}.png")
        plot_confusion_bars(top_df, f"Confusion Index — {label}", bar_path, base_class_order=all_labels)

    operator_cols = [c for c in wide_data[label].columns if c != RAW_ID_COLUMN]
    sample_df = wide_data[label].copy()
    sample_df["Unique Labels"] = sample_df[operator_cols].nunique(axis=1)
    sample_agreement_results[label] = sample_df

    summary = sample_df["Unique Labels"].value_counts().sort_index()
    print(f"\n  Per-sample label agreement ({label}):")
    for n_labels, n_samples in summary.items():
        print(f"    {n_samples} sample(s) have {n_labels} unique label(s) across operators")


### 3.2 Per-operator confusion profile

The chart above shows confusion *aggregated across all operators*. This breaks it down per
operator: for each interpreter, their personal average confusion rate (across all their
pairings), so an unusually high-confusion operator stands out individually rather than being
averaged away in the group total. Table in operator-ID order; chart ranked worst-first.


In [ ]:
operator_confusion = {}   # label -> per-operator mean confusion-index dataframe

for label, _ in VARIABLES:
    mats = confusion_matrices.get(label)
    if not mats:
        continue

    per_op_totals = {}
    for (op1, op2), mat in mats.items():
        off_diag_mean = (mat.values.sum() - np.trace(mat.values)) / (mat.shape[0] * (mat.shape[0] - 1))
        per_op_totals.setdefault(op1, []).append(off_diag_mean)
        per_op_totals.setdefault(op2, []).append(off_diag_mean)

    per_op_df = pd.DataFrame({
        "operator": list(per_op_totals.keys()),
        "Mean Confusion Index": [round(np.mean(v) * 100, 3) for v in per_op_totals.values()],
        "Pairings": [len(v) for v in per_op_totals.values()],
    }).set_index("operator")
    per_op_df = add_rank_column(per_op_df, "Mean Confusion Index", worst_is_high=True)

    operator_confusion[label] = per_op_df
    print(f"\n[{label}] Per-operator mean confusion index:")
    print(per_op_df)

    bar_path = os.path.join(OUTPUT_FOLDER, f"confusion_by_operator_{label}.png")
    plot_operator_bar(per_op_df, "Mean Confusion Index", f"Mean Confusion Index by Operator — {label}", bar_path, worst_is_high=True)


### 3.3 Majority-vote consensus & operator deviation

For each sample, the majority label across all operators acts as a practical consensus reference
(not "truth" -- just what most interpreters agreed on). This flags: (a) samples with no clear
majority, i.e. high genuine confusion, worth a group discussion; and (b) which operators most
often land on the minority label, i.e. individually diverge from the group.


In [ ]:
majority_consensus = {}   # label -> per-sample consensus dataframe
ambiguous_samples = {}    # label -> dataframe of samples with no clear majority (for export)
operator_deviation = {}   # label -> per-operator deviation-from-majority dataframe

for label, _ in VARIABLES:
    wide = wide_data[label]
    operator_cols = [c for c in wide.columns if c != RAW_ID_COLUMN]
    df = wide.copy()

    def majority_label(row):
        counts = row[operator_cols].value_counts()
        if len(counts) == 0:
            return None, 0.0
        top_label = counts.index[0]
        is_tie = (counts == counts.iloc[0]).sum() > 1
        share = counts.iloc[0] / counts.sum()
        return (None if is_tie else top_label), share

    majority_info = df[operator_cols].apply(majority_label, axis=1)
    df["Majority Label"] = [m[0] for m in majority_info]
    df["Majority Share"] = [round(m[1], 3) for m in majority_info]
    majority_consensus[label] = df[[RAW_ID_COLUMN, "Majority Label", "Majority Share"] + operator_cols]

    tied = df[df["Majority Label"].isna()]
    ambiguous_samples[label] = tied[[RAW_ID_COLUMN] + operator_cols]
    tied_ids = tied[RAW_ID_COLUMN].tolist()
    print(f"\n[{label}] Samples with no clear majority (every label appears equally often -- "
          f"flag for group discussion): {len(tied_ids)}")
    if tied_ids:
        shown = tied_ids[:15]
        suffix = f" ... and {len(tied_ids) - 15} more" if len(tied_ids) > 15 else ""
        print(f"  Sample IDs: {shown}{suffix}")
        print(f"  (Full list with each operator's label, per sample, is in the "
              f"'Ambiguous_{label}' sheet of the results workbook.)")

    deviation_counts = {op: 0 for op in operator_cols}
    valid_rows = df[df["Majority Label"].notna()]
    for op in operator_cols:
        deviation_counts[op] = (valid_rows[op] != valid_rows["Majority Label"]).sum()

    dev_df = pd.DataFrame({
        "operator": list(deviation_counts.keys()),
        "Disagrees with Majority": list(deviation_counts.values()),
        "Total Comparable Samples": len(valid_rows),
    }).set_index("operator")
    dev_df["Disagreement Rate (%)"] = round(100 * dev_df["Disagrees with Majority"] / dev_df["Total Comparable Samples"], 2)
    dev_df = add_rank_column(dev_df, "Disagreement Rate (%)", worst_is_high=True)
    operator_deviation[label] = dev_df

    print(f"\n[{label}] Per-operator disagreement with majority:")
    print(dev_df)

    bar_path = os.path.join(OUTPUT_FOLDER, f"majority_deviation_by_operator_{label}.png")
    plot_operator_bar(dev_df, "Disagreement Rate (%)", f"Disagreement with Majority by Operator — {label}", bar_path, worst_is_high=True)


## 4. Optional diagnostics

These two checks only run if the relevant column is configured **and** present in your data --
otherwise they're skipped with a short note, so the same notebook works cleanly whether or not a
given project's data includes them.


### 4.1 Vigilance / fatigue check (timestamp-based)

Checks whether disagreement with the majority label increases later in an operator's session --
evidence for (or against) limiting samples interpreted per day. Only runs if `TIMESTAMP_COLUMN`
is set in Section 1 and present in the data (only one of your two source tools currently records
this).


In [ ]:
if not TIMESTAMP_COLUMN or TIMESTAMP_COLUMN not in combined_df.columns:
    print(f"TIMESTAMP_COLUMN is not set or not present in this dataset -- skipping vigilance check. "
          f"(Set TIMESTAMP_COLUMN in Section 1 if this data source records interpretation timestamps.)")
else:
    ts_df = combined_df[[RAW_ID_COLUMN, "operator", TIMESTAMP_COLUMN]].dropna().copy()
    ts_df[TIMESTAMP_COLUMN] = pd.to_datetime(ts_df[TIMESTAMP_COLUMN])
    ts_df = ts_df.sort_values(["operator", TIMESTAMP_COLUMN])
    ts_df["session_position"] = ts_df.groupby("operator").cumcount()
    ts_df["session_length"] = ts_df.groupby("operator")["operator"].transform("count")
    ts_df["session_progress_bin"] = pd.qcut(
        ts_df["session_position"] / ts_df["session_length"], q=4,
        labels=["1st quarter", "2nd quarter", "3rd quarter", "4th quarter"],
    )

    primary_label = VARIABLES[0][0]
    consensus = majority_consensus.get(primary_label)
    if consensus is not None:
        merged = ts_df.merge(consensus[[RAW_ID_COLUMN, "Majority Label"]], on=RAW_ID_COLUMN, how="left")
        merged = merged.merge(
            combined_df[[RAW_ID_COLUMN, "operator", VARIABLES[0][1]]],
            on=[RAW_ID_COLUMN, "operator"], how="left",
        )
        merged["Disagrees with Majority"] = merged[VARIABLES[0][1]] != merged["Majority Label"]

        trend = merged.groupby("session_progress_bin", observed=True)["Disagrees with Majority"].mean() * 100
        print(f"Disagreement-with-majority rate by session progress ({primary_label}):")
        print(trend.round(2))
        if trend.iloc[-1] > trend.iloc[0]:
            print("\n  Disagreement rate is higher later in sessions -- possible fatigue/vigilance effect.")
        else:
            print("\n  No clear increase in disagreement later in sessions.")
    else:
        print("Majority-consensus data not available -- run Section 3.3 first.")


### 4.2 Confidence / difficulty cross-tab

Cross-tabulates interpreters' self-reported confidence or difficulty flags against their actual
disagreement with the majority label -- validates whether "I found this hard" lines up with
measured disagreement. Only runs if `CONFIDENCE_COLUMN` is set in Section 1 and present in the
data.


In [ ]:
if not CONFIDENCE_COLUMN or CONFIDENCE_COLUMN not in combined_df.columns:
    print(f"CONFIDENCE_COLUMN is not set or not present in this dataset -- skipping this check. "
          f"(Set CONFIDENCE_COLUMN in Section 1 if interpreters flag hard/low-confidence samples.)")
else:
    primary_label = VARIABLES[0][0]
    consensus = majority_consensus.get(primary_label)
    if consensus is not None:
        conf_df = combined_df[[RAW_ID_COLUMN, "operator", CONFIDENCE_COLUMN, VARIABLES[0][1]]].dropna()
        merged = conf_df.merge(consensus[[RAW_ID_COLUMN, "Majority Label"]], on=RAW_ID_COLUMN, how="left")
        merged["Disagrees with Majority"] = merged[VARIABLES[0][1]] != merged["Majority Label"]
        crosstab = merged.groupby(CONFIDENCE_COLUMN)["Disagrees with Majority"].agg(["mean", "count"])
        crosstab["mean"] = (crosstab["mean"] * 100).round(2)
        crosstab.columns = ["Disagreement Rate (%)", "N Samples"]
        print(f"Disagreement rate by self-reported {CONFIDENCE_COLUMN} ({primary_label}):")
        print(crosstab)
    else:
        print("Majority-consensus data not available -- run Section 3.3 first.")


## 5. Export consolidated results

All Gwet's AC1, Confusion Index, and diagnostic tables are written into **one Excel workbook**,
one sheet per result table, so everything is in a single file to share. The operator identity
mapping (Section 1.2) is kept in its own separate file, since it's sensitive and shouldn't travel
with shared results.


In [ ]:
results_path = os.path.join(OUTPUT_FOLDER, "calibration_results_gwet.xlsx")

with pd.ExcelWriter(results_path, engine="openpyxl") as writer:
    summary_rows = [{"Variable": label, "Operators": len([c for c in wide_data[label].columns if c != RAW_ID_COLUMN]),
                      "Samples": len(wide_data[label]),
                      "Mean Gwet's AC1": round(gwet_results[label]["Gwet's AC1"].mean(), 3) if label in gwet_results and not gwet_results[label].empty else None}
                     for label, _ in VARIABLES]
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

    for label, _ in VARIABLES:
        if label in gwet_results and not gwet_results[label].empty:
            gwet_results[label].to_excel(writer, sheet_name=f"Gwet_{label}"[:31], index=False)
        if label in operator_agreement_gwet:
            operator_agreement_gwet[label].to_excel(writer, sheet_name=f"GwetByOp_{label}"[:31])
        if label in confusion_results:
            confusion_results[label].to_excel(writer, sheet_name=f"ConfIndex_{label}"[:31], index=False)
        if label in operator_confusion:
            operator_confusion[label].to_excel(writer, sheet_name=f"ConfByOp_{label}"[:31])
        if label in operator_deviation:
            operator_deviation[label].to_excel(writer, sheet_name=f"MajorityDev_{label}"[:31])
        if label in sample_agreement_results:
            sample_agreement_results[label].to_excel(writer, sheet_name=f"SampleAgree_{label}"[:31], index=False)
        if label in ambiguous_samples and not ambiguous_samples[label].empty:
            ambiguous_samples[label].to_excel(writer, sheet_name=f"Ambiguous_{label}"[:31], index=False)
        if label in class_count_tables:
            class_count_tables[label].to_excel(writer, sheet_name=f"ClassCounts_{label}"[:31])

print(f"Consolidated results saved to: {results_path}")


### 5.1 Outputs summary


In [ ]:
print("=== Outputs produced ===\n")
print(f"1. Operator identity mapping (background, sensitive):\n   {identity_path}\n")
print(f"2. Consolidated results workbook (Gwet's AC1 + Confusion Index + diagnostics, all variables):\n   {results_path}\n")
print("3. Heatmap and bar chart images:")
for label, _ in VARIABLES:
    if label in gwet_results and not gwet_results[label].empty:
        print(f"   - gwet_heatmap_{label}.png")
    if label in confusion_results:
        print(f"   - confusion_index_{label}.png")
print(f"\nAll saved under: {OUTPUT_FOLDER}")
